# Matching c14_master_v08.xlsx site identifiers to SEAD_staging tbl_sites

The xlsx has `site_id` and `raa_id` columns. We match each of them separately against `tbl_sites.national_site_identifier` in the SEAD_staging database.


In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


## Load c14_master_v08.xlsx


In [11]:
excel_path = "../data/c14_master_v08.xlsx"
df = pd.read_excel(excel_path)
df[['site_id', 'raa_id']].head()


,site_id,raa_id
0,L1969:5267,Jörlanda 158
1,L1970:9431,Jörlanda 185
2,L1970:9431,Jörlanda 185
3,L1970:9431,Jörlanda 185
4,L1970:9431,Jörlanda 185


### site_id and raa_id overview


In [ ]:
print(f"{df['site_id'].notna().sum()} rows with site_id, {df['site_id'].nunique()} distinct values")
print(f"{df['raa_id'].notna().sum()} rows with raa_id, {df['raa_id'].nunique()} distinct values")


29963 rows with site_id, 6118 distinct values
28460 rows with raa_id, 5437 distinct values


## Connect to sead_staging database


In [2]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DB_HOST = os.environ["DB_HOST"]
DB_PORT = os.environ["DB_PORT"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


## Load tbl_sites


In [ ]:
sites = pd.read_sql(
    'select site_id, national_site_identifier, site_name from public.tbl_sites order by site_id',
    engine,
)
print(f'{len(sites)} rows in tbl_sites')
print(f"{sites['national_site_identifier'].notna().sum()} rows with national_site_identifier, "
      f"{sites['national_site_identifier'].nunique()} distinct values")
sites.head()


3462 rows in tbl_sites
1500 rows with national_site_identifier, 1440 distinct values


,site_id,national_site_identifier,site_name
0,1,Slöinge 114,Slöinge Raä 114
1,2,Skrea 194:1,Skrea Raä 194
2,3,Skrea 177:1,Skrea Raä 177
3,4,Onsala Raä 327:1,Onsala 327
4,5,Vallda 293:1,Vallda Raä 293


## Match site_id (xlsx) against national_site_identifier (tbl_sites)


In [ ]:
site_id_matches = (
    df[['site_id']].dropna().drop_duplicates()
    .merge(
        sites.rename(columns={'site_id': 'sead_site_id'}),
        left_on='site_id',
        right_on='national_site_identifier',
        how='inner',
    )
)
print(f"{site_id_matches['site_id'].nunique()} of {df['site_id'].nunique()} distinct site_id values "
      "matched a national_site_identifier")
site_id_matches


42 of 6118 distinct site_id values matched a national_site_identifier


,site_id,sead_site_id,national_site_identifier,site_name
0,L1991:2350,3837,L1991:2350,Åby
1,L1988:2436,6382,L1988:2436,Helsingborg 42:1
2,L1989:8982,6392,L1989:8982,Helsingborg 226:1
3,L1983:2662,6376,L1983:2662,Åker 270:1
4,L2011:1331,6412,L2011:1331,Motala 173:1
5,L2015:2168,6421,L2015:2168,Sigtuna 195:1
6,L1959:7102,6363,L1959:7102,Algutsboda 79:1
7,L2017:1568,6424,L2017:1568,Adelsö 119:1
8,L1989:3865,5790,L1989:3865,Falsterbo kyrka
9,L1989:3865,6390,L1989:3865,Falsterbo 15:1


## Match raa_id (xlsx) against national_site_identifier (tbl_sites)


In [ ]:
raa_id_matches = (
    df[['raa_id']].dropna().drop_duplicates()
    .merge(
        sites.rename(columns={'site_id': 'sead_site_id'}),
        left_on='raa_id',
        right_on='national_site_identifier',
        how='inner',
    )
)
print(f"{raa_id_matches['raa_id'].nunique()} of {df['raa_id'].nunique()} distinct raa_id values "
      "matched a national_site_identifier")
raa_id_matches


106 of 5437 distinct raa_id values matched a national_site_identifier


,raa_id,sead_site_id,national_site_identifier,site_name
0,Odensala 6,329,Odensala 6,Odensala 6 (386)
1,Strängnäs 266,3580,Strängnäs 266,Lunda omr. B
2,Sigtuna 195,3463,Sigtuna 195,St. Gatan. Kv. Handelsmannen 8-9
3,Sigtuna 195,3615,Sigtuna 195,kv. Trädgårdsmästaren
4,Sigtuna 195,3628,Sigtuna 195,kv Trädgårdsmästaren
...,...,...,...,...
118,Norra Nöbbelöv 13,3540,Norra Nöbbelöv 13,Norra Nöbbelöv
119,Grevie 363,3722,Grevie 363,Grevie 363
120,Linköping 188,3585,Linköping 188,Linköping 188
121,Norrsunda 167,3538,Norrsunda 167,Norrsunda 167


## Understanding of features linked to sites in SEAD
as the Strucke data contains columns such as _context_id_ and _context_type_ that refer to data that in SEAD could be allocated in _tbl_features_ and _tbl_feature_types_, a follow-up question that comes is: how would these be linked in SEAD after ingestion?

That's why now for the EDA we will retrieve the data from a sql query that connects the _tbl_sites_ with the tables with _feature_ content to see both the path and draw conclusions of their connection.

In [8]:
query = '''
SELECT s.site_id, sg.sample_group_id, ps.physical_sample_id, psf.physical_sample_feature_id,
f.feature_id, ft.feature_type_id, s.latitude_dd, s.longitude_dd, s.national_site_identifier, 
s.site_name, s.site_description,  sg.method_id, sg.sample_group_name, sg.sample_group_description,
ps.sample_name, ps.date_sampled, f.feature_name, f.feature_description, ft.feature_type_name,
ft.feature_type_description, s.date_updated as "site_date_update", 
sg.date_updated as "sample_group_date_updated", ps.date_updated as "physical_sample_date_updated",
psf.date_updated as "physical_sample_feature_date_updated", f.date_updated as "features_date_updated",
ft.date_updated as "feature_types_date_updated"
FROM tbl_sites s
join tbl_sample_groups sg on s.site_id = sg.site_id
join tbl_physical_samples ps on sg.sample_group_id = ps.sample_group_id
join tbl_physical_sample_features psf on ps.physical_sample_id = psf.physical_sample_id
join tbl_features f on psf.feature_id = f.feature_id
left join tbl_feature_types ft on f.feature_type_id = ft.feature_type_id
'''

sites_features = pd.read_sql(
    query, engine,
)

sites_features

,site_id,sample_group_id,physical_sample_id,physical_sample_feature_id,feature_id,feature_type_id,latitude_dd,longitude_dd,national_site_identifier,site_name,site_description,method_id,sample_group_name,sample_group_description,sample_name,date_sampled,feature_name,feature_description,feature_type_name,feature_type_description,site_date_update,sample_group_date_updated,physical_sample_date_updated,physical_sample_feature_date_updated,features_date_updated,feature_types_date_updated
0,2,2,2,360,323,10,56.585000,12.597500,Skrea 194:1,Skrea Raä 194,NaN,60,Features,None,PM7511,NaN,5798,NaN,Cesspit,Pit or structure used for waste products and/o...,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2012-11-01 15:05:01.075365+01:00
1,3,3,3,1,1,26,56.880556,12.616944,Skrea 177:1,Skrea Raä 177,NaN,60,House 14,None,5854,NaN,5854,NaN,Post hole,A depression interpreted as having been made b...,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-04-16 16:45:29.540689+02:00
2,4,5,10,136,257,6,57.380583,12.018033,Onsala Raä 327:1,Onsala 327,NaN,60,A5518,None,99_0182:0001,NaN,A5518,Uncertain,Grave/burial,Burial of undefined type,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2012-11-01 14:51:37.977054+01:00
3,5,7,14,183,345,7,57.476389,11.998056,Vallda 293:1,Vallda Raä 293,NaN,60,Feature 2628,None,B,NaN,2628,"Oven residue, on a fragmentary settlement. Pos...",Pit,Natural or man made hole or depression of unsp...,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2012-11-01 14:55:28.292677+01:00
4,4,5,16,152,293,6,57.380583,12.018033,Onsala Raä 327:1,Onsala 327,NaN,60,A5518,None,99_0182:0034,NaN,A5518,Uncertain,Grave/burial,Burial of undefined type,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2012-11-01 14:51:37.977054+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7963,3486,11849,49578,12830,4817,551,56.022894,14.607754,Mjällby 61?,Siretorp,NaN,173,KFL 5@Kärl,None,5@123@Kärl,NaN,Feature,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
7964,3420,11850,49579,12831,4818,551,55.613738,14.287926,Rörum 15,Vik,NaN,173,KFL 4@Lampa,None,4@53@Lampa,NaN,Feature,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
7965,3420,11851,49580,12832,4818,551,55.613738,14.287926,Rörum 15,Vik,NaN,173,KFL 3@Kärl,None,3@52@Kärl,NaN,Feature,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
7966,3420,11852,49581,12833,4818,551,55.613738,14.287926,Rörum 15,Vik,NaN,173,KFL 2@Kärl,None,2@51@Kärl,NaN,Feature,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00


In [10]:
query_outer = '''
SELECT s.site_id, sg.sample_group_id, ps.physical_sample_id, psf.physical_sample_feature_id,
f.feature_id, ft.feature_type_id, s.latitude_dd, s.longitude_dd, s.national_site_identifier, 
s.site_name, s.site_description,  sg.method_id, sg.sample_group_name, sg.sample_group_description,
ps.sample_name, ps.date_sampled, f.feature_name, f.feature_description, ft.feature_type_name,
ft.feature_type_description, s.date_updated as "site_date_update", 
sg.date_updated as "sample_group_date_updated", ps.date_updated as "physical_sample_date_updated",
psf.date_updated as "physical_sample_feature_date_updated", f.date_updated as "features_date_updated",
ft.date_updated as "feature_types_date_updated"
FROM tbl_sites s
full outer join tbl_sample_groups sg on s.site_id = sg.site_id
full outer join tbl_physical_samples ps on sg.sample_group_id = ps.sample_group_id
full outer join tbl_physical_sample_features psf on ps.physical_sample_id = psf.physical_sample_id
full outer join tbl_features f on psf.feature_id = f.feature_id
left join tbl_feature_types ft on f.feature_type_id = ft.feature_type_id
'''

sites_features_outer = pd.read_sql(
    query_outer, engine,
)

sites_features_outer

,site_id,sample_group_id,physical_sample_id,physical_sample_feature_id,feature_id,feature_type_id,latitude_dd,longitude_dd,national_site_identifier,site_name,site_description,method_id,sample_group_name,sample_group_description,sample_name,date_sampled,feature_name,feature_description,feature_type_name,feature_type_description,site_date_update,sample_group_date_updated,physical_sample_date_updated,physical_sample_feature_date_updated,features_date_updated,feature_types_date_updated
0,1.0,1.0,1.0,NaN,NaN,NaN,56.866389,12.668333,Slöinge 114,Slöinge Raä 114,NaN,81.0,Survey,None,96_0010:0299,NaN,NaN,NaN,NaN,NaN,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,None,None,None
1,2.0,2.0,2.0,360.0,323.0,10.0,56.585000,12.597500,Skrea 194:1,Skrea Raä 194,NaN,60.0,Features,None,PM7511,NaN,5798,NaN,Cesspit,Pit or structure used for waste products and/o...,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2012-11-01 15:05:01.075365+01:00
2,3.0,3.0,3.0,1.0,1.0,26.0,56.880556,12.616944,Skrea 177:1,Skrea Raä 177,NaN,60.0,House 14,None,5854,NaN,5854,NaN,Post hole,A depression interpreted as having been made b...,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-04-16 16:45:29.540689+02:00
3,1.0,1.0,4.0,NaN,NaN,NaN,56.866389,12.668333,Slöinge 114,Slöinge Raä 114,NaN,81.0,Survey,None,96_0010:0146,NaN,NaN,NaN,NaN,NaN,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,None,None,None
4,2.0,4.0,5.0,NaN,NaN,NaN,56.585000,12.597500,Skrea 194:1,Skrea Raä 194,NaN,81.0,Survey,None,1191,1994,NaN,NaN,NaN,NaN,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,2013-05-13 11:29:56.070308+02:00,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44455,NaN,NaN,NaN,NaN,4302.0,551.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,8184,NaN,Settlement site,A site previously inhabited where worked objec...,None,None,None,None,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
44456,NaN,NaN,NaN,NaN,3992.0,551.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,"4093:28, A1:34",NaN,Settlement site,A site previously inhabited where worked objec...,None,None,None,None,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
44457,NaN,NaN,NaN,NaN,3994.0,551.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,"4093:12, A1:24",NaN,Settlement site,A site previously inhabited where worked objec...,None,None,None,None,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
44458,NaN,NaN,NaN,NaN,4356.0,551.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,7107,NaN,Settlement site,A site previously inhabited where worked objec...,None,None,None,None,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00


from this we see that not all sites are connected to features

### Check if the strucke sites can appear in those with features
#### first with the Lämningsnummer (site_id in strucke)

In [13]:
site_feature_id_matches = (
    df[['site_id']].dropna().drop_duplicates()
    .merge(
        sites_features.rename(columns={'site_id': 'sead_site_id'}),
        left_on='site_id',
        right_on='national_site_identifier',
        how='inner',
    )
)
print(f"{site_feature_id_matches['site_id'].nunique()} of {df['site_id'].nunique()} distinct site_id values "
      "matched a national_site_identifier")
site_feature_id_matches

0 of 6118 distinct site_id values matched a national_site_identifier


,site_id,sead_site_id,sample_group_id,physical_sample_id,physical_sample_feature_id,feature_id,feature_type_id,latitude_dd,longitude_dd,national_site_identifier,site_name,site_description,method_id,sample_group_name,sample_group_description,sample_name,date_sampled,feature_name,feature_description,feature_type_name,feature_type_description,site_date_update,sample_group_date_updated,physical_sample_date_updated,physical_sample_feature_date_updated,features_date_updated,feature_types_date_updated


#### Now with the RAÄ_id from strucke matching the national site identifier from sead

In [14]:
raa_feature_id_matches = (
    df[['raa_id']].dropna().drop_duplicates()
    .merge(
        sites_features.rename(columns={'site_id': 'sead_site_id'}),
        left_on='raa_id',
        right_on='national_site_identifier',
        how='inner',
    )
)
print(f"{raa_feature_id_matches['raa_id'].nunique()} of {df['raa_id'].nunique()} distinct raa_id values "
      "matched a national_site_identifier")
raa_feature_id_matches

94 of 5437 distinct raa_id values matched a national_site_identifier


,raa_id,sead_site_id,sample_group_id,physical_sample_id,physical_sample_feature_id,feature_id,feature_type_id,latitude_dd,longitude_dd,national_site_identifier,site_name,site_description,method_id,sample_group_name,sample_group_description,sample_name,date_sampled,feature_name,feature_description,feature_type_name,feature_type_description,site_date_update,sample_group_date_updated,physical_sample_date_updated,physical_sample_feature_date_updated,features_date_updated,feature_types_date_updated
0,Strängnäs 266,3580,8567,46296,9725,3961,551,59.368600,16.953009,Strängnäs 266,Lunda omr. B,NaN,173,KFL 4765@Sländtrissa &amp; vävtyngd,None,4765@5@Sländtrissa &amp; vävtyngd,NaN,20769,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
1,Strängnäs 266,3580,8568,46297,9726,3962,551,59.368600,16.953009,Strängnäs 266,Lunda omr. B,NaN,173,KFL 4764@Sländtrissa &amp; vävtyngd,None,4764@4@Sländtrissa &amp; vävtyngd,NaN,47619,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
2,Strängnäs 266,3580,8569,46298,9727,3963,551,59.368600,16.953009,Strängnäs 266,Lunda omr. B,NaN,173,KFL 4763@Sländtrissa &amp; vävtyngd,None,4763@3@Sländtrissa &amp; vävtyngd,NaN,11978,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
3,Strängnäs 266,3580,8570,46299,9728,3964,551,59.368600,16.953009,Strängnäs 266,Lunda omr. B,NaN,173,KFL 4762@Blästermunstycke,None,4762@2@Blästermunstycke,NaN,21183,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
4,Strängnäs 266,3580,8571,46300,9729,3965,551,59.368600,16.953009,Strängnäs 266,Lunda omr. B,NaN,173,KFL 4761@Blästermunstycke,None,4761@1@Blästermunstycke,NaN,14141,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
983,Norrsunda 167,3538,10620,48349,11748,4586,551,59.582382,17.900611,Norrsunda 167,Norrsunda 167,NaN,173,KFL 1613@Kärl,None,1613@5@Kärl,NaN,Feature,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
984,Norrsunda 167,3538,10621,48350,11749,4586,551,59.582382,17.900611,Norrsunda 167,Norrsunda 167,NaN,173,KFL 1612@Kärl,None,1612@4@Kärl,NaN,Feature,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2019-12-20 14:45:52.481448+01:00
985,Norrsunda 167,3538,10622,48351,11750,4586,551,59.582382,17.900611,Norrsunda 167,Norrsunda 167,NaN,173,KFL 1611@Kärl,None,1611@3@Kärl,NaN,Feature,NaN,Settlement site,A site previously inhabited where worked objec...,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01-09 09:37:49.705454+01:00,2020-01

this means that some of the sites to be ingested in strucke already exist in sead, having a link to the features. However, the features get linked through _tbl_sample_groups_ and _tbl_physical_samples_, which at first makes me think of different possibilities to do this:

1. link the data for sites considering this raä_id as a valid national identifier.
    Pros:
    - when looking in the sead browser, the filter by site would correctly give all the information about the site.
    - the features can be connected through the sample tables to an already existing site
    Cons:
    - We would need to ingest the lämningsnummer in another column (e.g. in the site description as suggested by Riia)
    - It wouldn't make it easy for a user to search for the information of a site if they only have the lämningsnummer

2. Link the data for the sites considering the lämningsnummer.
    Pros:
    - would make the mapping easier only to rely on one column for the national identifier.
    - ingesting all the sites as new (because no links with lämningsnummer to the site) seems to be an easier approach for the ingestion than looking for links and specify them in ShapeShifter (maybe not)
    Cons:
    - If a user needs to look for a site and they just know the Raä_id, then they won't find all data in the browser (unless the sites data gets grouped by the coordinates in which case I assume it would be the same)
    - somehow there should be a link between data that references the same site, it would create another problem to solve

3. Ingest the data twice, one with national_site_identifier as the raä_id and the other one with the lämningsnummer.
    Pros:
    - User could look up for a site and find all the information, only if they have the raä_id. If they have lämningsnummer, they would find the information about strucke data only.
    - Once found the way of mapping the strucke data to sead, it would be straight forward ingestion of similar entities with different values linked to national site identifier.
    Cons:
    - Duplicated data for the values, could pose a problem (I'm not sure) for users if they could end up getting the duplicated data and think there is double the amount of values measured.
    